In [12]:
"""
TOP1模型 (Gp_GP_Matern_0.5_K6_Comb17) 完整CDF分析程序 v5
基于v4修改：
  ✅ 04_Error_PDF：增加峰值标注、分区着色、统计文本框、模式线
  ✅ 01_CDF：增加关键百分位数标注、统计文本框、网格分区
  ✅ 02_Scatter：增加±10%/±20%误差带、R²/RMSE文本框、密度着色
  ✅ 03_Exceedance：增加"达标区"/"警戒区"分区着色、关键点标注
  ✅ 05_CumError：增加洛伦兹系数、基尼系数标注、关键点标注
  ✅ 06_Quantile：增加拟合优度R²、偏离区域着色、分位数分段着色
  ✅ 07_Histogram：增加正态拟合曲线、分位数线、统计文本框
  ✅ 08_Radar：增加数值标签、得分等级背景着色
  ✅ 09_ThresholdBar：增加达标/不达标颜色编码、目标线、差距标注
  ✅ 10_Boxplot：增加均值±std标注、样本量标注、分位数注释

输出目录: D:\博二上\模型分析可视化\CDF分析6
"""

import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family']        = 'Arial'
matplotlib.rcParams['pdf.fonttype']       = 42
matplotlib.rcParams['ps.fonttype']        = 42
matplotlib.rcParams['axes.unicode_minus'] = False

# ====================== 配置参数 ======================
CONFIG = {
    'pkl_file': (
        r"D:\博一下\第一章主动学习\高斯过程组_结果_v3.5-Extended"
        r"\top30_models\model_objects"
        r"\rank01_Gp_GP_Matern_0.5_K6_Comb17_model.pkl"
    ),
    'data_file': (
        r"D:\博一下\第一章主动学习"
        r"\Nb-Si数据库10.18-26个-4种参数-成分-性能-Si小于10的全部去掉.xlsx"
    ),
    'output_path':  r"D:\博二上\模型分析可视化\CDF分析6",
    'target_col':   'KQ',
    'random_state': 2023,
    'max_error':        50,
    'error_thresholds': [5, 10, 15, 20, 25, 30],
    'figure_dpi':       300,
    'bootstrap_n':  2000,
    'bootstrap_ci': 0.95,
}

COLORS = {
    'All':   '#1f77b4',
    'Train': '#2ca02c',
    'Test':  '#d62728',
}

# ====================== 目录创建 ======================
def setup_output_directory():
    out = CONFIG['output_path']
    for sub in [
        'figures', 'figures/individual_models',
        'data', 'data/individual_errors',
        'statistics', 'excel_for_origin',
    ]:
        os.makedirs(os.path.join(out, sub), exist_ok=True)
    print(f"✓ 输出目录已创建: {out}")
    return out

# ====================== GP预测 ======================
def gp_predict(model, scaler, X_raw):
    X_sc = scaler.transform(X_raw)
    try:
        y_mean, y_std = model.predict(X_sc, return_std=True)
        return y_mean, y_std
    except Exception:
        y_mean = model.predict(X_sc)
        return y_mean, np.zeros_like(y_mean)

# ====================== 数据加载 ======================
def load_gp_model_and_data():
    print("=== 加载TOP1高斯过程模型和数据 ===")
    with open(CONFIG['pkl_file'], 'rb') as f:
        md = pickle.load(f)

    gp_model      = md['model']
    scaler        = md['scaler']
    feature_names = md['features']
    model_name    = md['model_name']

    print(f"✓ 模型类型  : {model_name}")
    print(f"✓ 特征列表  : {feature_names}")
    print(f"✓ Test R²   : {md.get('test_r2_true', 'N/A')}")

    df       = pd.read_excel(CONFIG['data_file'])
    df_valid = df[feature_names + [CONFIG['target_col']]].dropna()
    df_valid = df_valid[
        df_valid[CONFIG['target_col']] > 0].reset_index(drop=True)
    X_all = df_valid[feature_names].values
    y_all = df_valid[CONFIG['target_col']].values
    print(f"✓ 有效样本  : {len(df_valid)}")

    def create_stratify_bins(y, n_bins=5):
        pe = np.linspace(0, 100, n_bins + 1)
        be = np.percentile(y, pe)
        be[0] = -np.inf; be[-1] = np.inf
        return np.digitize(y, be) - 1

    strat = create_stratify_bins(y_all)
    idx   = np.arange(len(y_all))
    tr_idx, te_idx, _, _ = train_test_split(
        idx, idx, test_size=0.2, stratify=strat, random_state=2023)
    print(f"✓ 训练集: {len(tr_idx)}  测试集: {len(te_idx)}")

    return {
        'gp_model':      gp_model,
        'scaler':        scaler,
        'feature_names': feature_names,
        'model_name':    model_name,
        'model_data':    md,
        'X_all':         X_all,
        'y_all':         y_all,
        'train_idx':     tr_idx,
        'test_idx':      te_idx,
    }

# ====================== 误差计算 ======================
def calc_stats(y_true, y_pred, y_std):
    safe  = np.where(np.abs(y_true) < 1e-6, 1e-6, y_true)
    rel   = np.abs((y_true - y_pred) / safe) * 100
    res   = y_true - y_pred
    cov95 = np.mean(np.abs(res) <= 1.96 * y_std)
    cov68 = np.mean(np.abs(res) <= y_std)
    return {
        'y_true': y_true, 'y_pred': y_pred, 'y_std': y_std,
        'residuals': res,
        'std_residuals': res / np.std(res) if np.std(res) > 0 else res,
        'relative_errors': rel,
        'absolute_errors': np.abs(res),
        'r2':   r2_score(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae':  mean_absolute_error(y_true, y_pred),
        'mean_rel_err':   np.mean(rel),
        'median_rel_err': np.median(rel),
        'std_rel_err':    np.std(rel),
        'cov95': cov95, 'cov68': cov68,
        'f5':  np.mean(rel <=  5), 'f10': np.mean(rel <= 10),
        'f15': np.mean(rel <= 15), 'f20': np.mean(rel <= 20),
        'f25': np.mean(rel <= 25), 'f30': np.mean(rel <= 30),
    }

def calculate_errors(bundle):
    print("\n=== 计算预测误差 ===")
    m, sc = bundle['gp_model'], bundle['scaler']
    X, y  = bundle['X_all'], bundle['y_all']
    ti, ei = bundle['train_idx'], bundle['test_idx']

    ym_a, ys_a = gp_predict(m, sc, X)
    ym_t, ys_t = gp_predict(m, sc, X[ti])
    ym_e, ys_e = gp_predict(m, sc, X[ei])

    sa = calc_stats(y,     ym_a, ys_a)
    st = calc_stats(y[ti], ym_t, ys_t)
    se = calc_stats(y[ei], ym_e, ys_e)

    for lbl, s in [('全部数据', sa), ('训练集', st), ('测试集', se)]:
        print(f"  {lbl}  R²={s['r2']:.4f}  MAE={s['mae']:.4f}  "
              f"MRE={s['mean_rel_err']:.2f}%  F10={s['f10']:.3f}")
    return sa, st, se

# ====================== Bootstrap CDF ======================
def bootstrap_cdf(errors, x_grid, n_boot=None, ci=None):
    if n_boot is None: n_boot = CONFIG['bootstrap_n']
    if ci is None:     ci = CONFIG['bootstrap_ci']
    alpha = 1 - ci
    n     = len(errors)
    boot_cdfs = np.zeros((n_boot, len(x_grid)))
    for b in range(n_boot):
        samp = np.random.choice(errors, size=n, replace=True)
        samp_sorted = np.sort(samp)
        cum = np.arange(1, n + 1) / n
        boot_cdfs[b] = np.interp(x_grid, samp_sorted, cum, left=0, right=1)
    lo = np.percentile(boot_cdfs, alpha/2*100,       axis=0)
    hi = np.percentile(boot_cdfs, (1-alpha/2)*100,   axis=0)
    return lo, hi

def _plot_cdf_with_ci(ax, errors, x_grid, color, label,
                      ls='-', lw=2.5, alpha_band=0.15, n_boot=None):
    ef  = errors[errors <= CONFIG['max_error']]
    es  = np.sort(ef)
    cum = np.arange(1, len(es)+1) / len(es)
    yi  = np.interp(x_grid, es, cum, left=0, right=1)
    ax.plot(x_grid, yi, color=color, linewidth=lw,
            linestyle=ls, label=label, zorder=4)
    lo, hi = bootstrap_cdf(ef, x_grid, n_boot)
    ax.fill_between(x_grid, lo, hi, color=color,
                    alpha=alpha_band, zorder=3)

# ====================== 辅助工具函数 ======================
def add_stats_textbox(ax, text, x=0.97, y=0.05, fontsize=8.5,
                      ha='right', va='bottom',
                      boxstyle='round,pad=0.4', alpha=0.85):
    """在ax上添加统计文本框"""
    ax.text(x, y, text, transform=ax.transAxes,
            fontsize=fontsize, ha=ha, va=va,
            bbox=dict(boxstyle=boxstyle, facecolor='lightyellow',
                      edgecolor='gray', alpha=alpha))

def compute_gini(errors):
    """计算基尼系数（误差集中度）"""
    ae = np.sort(np.abs(errors))
    n  = len(ae)
    idx = np.arange(1, n + 1)
    return (2 * np.sum(idx * ae) / (n * np.sum(ae))) - (n + 1) / n

# ====================== Excel保存（与v4相同，略做精简）======================
def _save_excel_for_origin(bundle, sa, st, se, excel_dir, model_name):
    """Excel保存逻辑与v4完全相同，此处保留以保证完整性"""
    print("  生成Origin可用Excel文件...")
    MAX_E    = CONFIG['max_error']
    x_common = np.linspace(0, MAX_E, 1000)
    N_BOOT   = CONFIG['bootstrap_n']
    thr_list = CONFIG['error_thresholds']

    # 01 CDF数据
    f01 = os.path.join(excel_dir, '01_CDF_Data.xlsx')
    with pd.ExcelWriter(f01, engine='openpyxl') as w:
        cdf_interp = {'Error_Percent': x_common}
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            ef  = s['relative_errors'][s['relative_errors'] <= MAX_E]
            es  = np.sort(ef)
            cum = np.arange(1, len(es)+1) / len(es)
            yi  = np.interp(x_common, es, cum, left=0, right=1)
            cdf_interp[f'{lbl}_CumProb'] = yi
            lo, hi = bootstrap_cdf(ef, x_common, N_BOOT)
            cdf_interp[f'{lbl}_CI_Lower_{int(CONFIG["bootstrap_ci"]*100)}'] = lo
            cdf_interp[f'{lbl}_CI_Upper_{int(CONFIG["bootstrap_ci"]*100)}'] = hi
        pd.DataFrame(cdf_interp).to_excel(
            w, sheet_name='CDF_with_Bootstrap_CI', index=False)
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            ef  = s['relative_errors'][s['relative_errors'] <= MAX_E]
            es  = np.sort(ef)
            cum = np.arange(1, len(es)+1) / len(es)
            pd.DataFrame({'Error_Percent': es, 'Cumulative_Prob': cum}
                         ).to_excel(w, sheet_name=f'CDF_Raw_{lbl}', index=False)
        rows = []
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            rows.append({
                'Dataset': lbl, 'Model': model_name, 'N': len(s['y_true']),
                'R2': s['r2'], 'RMSE': s['rmse'], 'MAE': s['mae'],
                'MRE': s['mean_rel_err'], 'MedRE': s['median_rel_err'],
                'F5': s['f5'], 'F10': s['f10'], 'F15': s['f15'],
                'F20': s['f20'], 'F25': s['f25'], 'F30': s['f30'],
                'GP_Cov95': s['cov95'], 'GP_Cov68': s['cov68'],
            })
        pd.DataFrame(rows).to_excel(w, sheet_name='Model_Stats', index=False)
    print(f"    ✓ 01_CDF_Data.xlsx")

    # 保存其余Excel（02~11），与v4完全相同
    f02 = os.path.join(excel_dir, '02_Radar_Chart_Data.xlsx')
    with pd.ExcelWriter(f02, engine='openpyxl') as w:
        radar_rows = []
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            radar_rows.append({
                'Dataset': lbl, 'R²': s['r2'], 'F5%': s['f5'],
                'F10%': s['f10'], 'F15%': s['f15'],
                'F20%': s['f20'], 'GP_Cov95%': s['cov95'],
            })
        pd.DataFrame(radar_rows).to_excel(w, sheet_name='Radar_Data', index=False)
    print(f"    ✓ 02_Radar_Chart_Data.xlsx")

    f03 = os.path.join(excel_dir, '03_Error_Histograms.xlsx')
    with pd.ExcelWriter(f03, engine='openpyxl') as w:
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            ef  = s['relative_errors'][s['relative_errors'] <= 30]
            cnt, edges = np.histogram(ef, bins=30)
            centers    = (edges[:-1] + edges[1:]) / 2
            pd.DataFrame({
                'Bin_Centers': centers, 'Counts': cnt,
                'Bin_Left': edges[:-1], 'Bin_Right': edges[1:],
                'Mean_Line_X':   [np.mean(ef)] * len(centers),
                'Median_Line_X': [np.median(ef)] * len(centers),
            }).to_excel(w, sheet_name=f'Histogram_{lbl}', index=False)
    print(f"    ✓ 03_Error_Histograms.xlsx")

    f04 = os.path.join(excel_dir, '04_Prediction_Scatter.xlsx')
    with pd.ExcelWriter(f04, engine='openpyxl') as w:
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            pd.DataFrame({
                'Actual': s['y_true'], 'Predicted': s['y_pred'],
                'Residuals': s['residuals'], 'Pred_Std': s['y_std'],
                'Rel_Error_Pct': s['relative_errors'],
            }).to_excel(w, sheet_name=f'Scatter_{lbl}', index=False)
            mn_v = min(s['y_true'].min(), s['y_pred'].min())
            mx_v = max(s['y_true'].max(), s['y_pred'].max())
            pd.DataFrame({'Perfect_X': [mn_v, mx_v], 'Perfect_Y': [mn_v, mx_v]}
                         ).to_excel(w, sheet_name=f'PerfectLine_{lbl}', index=False)
    print(f"    ✓ 04_Prediction_Scatter.xlsx")

    f05 = os.path.join(excel_dir, '05_GP_Uncertainty.xlsx')
    with pd.ExcelWriter(f05, engine='openpyxl') as w:
        unc_rows = []
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            sidx = np.argsort(s['y_pred'])
            pd.DataFrame({
                'Pred_Mean':   s['y_pred'][sidx],
                'True_Value':  s['y_true'][sidx],
                'Pred_Std':    s['y_std'][sidx],
                'CI_Upper_95': (s['y_pred'] + 1.96*s['y_std'])[sidx],
                'CI_Lower_95': (s['y_pred'] - 1.96*s['y_std'])[sidx],
            }).to_excel(w, sheet_name=f'Uncertainty_{lbl}', index=False)
            unc_rows.append({
                'Dataset': lbl,
                'Mean_Pred_Std':   np.mean(s['y_std']),
                'Coverage_95pct':  s['cov95'],
                'Coverage_68pct':  s['cov68'],
            })
        pd.DataFrame(unc_rows).to_excel(
            w, sheet_name='Uncertainty_Summary', index=False)
    print(f"    ✓ 05_GP_Uncertainty.xlsx")

    f06 = os.path.join(excel_dir, '06_Threshold_Bar_Data.xlsx')
    with pd.ExcelWriter(f06, engine='openpyxl') as w:
        bar_rows = []
        for thr in thr_list:
            bar_rows.append({
                'Threshold_%': thr, 'Label': f'F{thr}%',
                'All_Pct': sa[f'f{thr}']*100,
                'Train_Pct': st[f'f{thr}']*100,
                'Test_Pct':  se[f'f{thr}']*100,
            })
        pd.DataFrame(bar_rows).to_excel(
            w, sheet_name='Threshold_Rates', index=False)
        wide_rows = []
        for lbl, s in [('Train', st), ('Test', se), ('All', sa)]:
            row = {'Dataset': lbl}
            for thr in thr_list:
                row[f'F{thr}%'] = s[f'f{thr}'] * 100
            wide_rows.append(row)
        pd.DataFrame(wide_rows).to_excel(
            w, sheet_name='Wide_Format_for_Origin', index=False)
    print(f"    ✓ 06_Threshold_Bar_Data.xlsx")

    f07 = os.path.join(excel_dir, '07_Boxplot_Data.xlsx')
    with pd.ExcelWriter(f07, engine='openpyxl') as w:
        long_rows = []
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            for v in s['relative_errors']:
                long_rows.append({'Dataset': lbl, 'Relative_Error_%': v})
        pd.DataFrame(long_rows).to_excel(
            w, sheet_name='Raw_Errors_Long', index=False)
        box_sum = []
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            rel = s['relative_errors']
            q1, q3 = np.percentile(rel, 25), np.percentile(rel, 75)
            box_sum.append({
                'Dataset': lbl, 'Min': rel.min(), 'Q1': q1,
                'Median': np.median(rel), 'Q3': q3, 'Max': rel.max(),
                'Mean': rel.mean(), 'Std': rel.std(), 'IQR': q3-q1,
            })
        pd.DataFrame(box_sum).to_excel(
            w, sheet_name='Boxplot_Summary', index=False)
    print(f"    ✓ 07_Boxplot_Data.xlsx")

    f08 = os.path.join(excel_dir, '08_Exceedance_Data.xlsx')
    with pd.ExcelWriter(f08, engine='openpyxl') as w:
        exceedance_dict = {'Error_Percent': x_common}
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            ef  = s['relative_errors'][s['relative_errors'] <= MAX_E]
            es  = np.sort(ef)
            cum = np.arange(1, len(es)+1) / len(es)
            yi  = np.interp(x_common, es, cum, left=0, right=1)
            exceedance_dict[f'{lbl}_Exceedance'] = 1 - yi
            lo, hi = bootstrap_cdf(ef, x_common, N_BOOT)
            exceedance_dict[f'{lbl}_Exceed_CI_Upper'] = 1 - lo
            exceedance_dict[f'{lbl}_Exceed_CI_Lower'] = 1 - hi
        pd.DataFrame(exceedance_dict).to_excel(
            w, sheet_name='Exceedance_Probability', index=False)
        exceed_rows = []
        for thr in [5, 10, 15, 20, 25, 30]:
            row = {'Threshold_%': thr}
            for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
                row[f'{lbl}_Exceedance_%'] = (1 - s[f'f{thr}']) * 100
            exceed_rows.append(row)
        pd.DataFrame(exceed_rows).to_excel(
            w, sheet_name='Key_Threshold_Exceedance', index=False)
    print(f"    ✓ 08_Exceedance_Data.xlsx")

    f09 = os.path.join(excel_dir, '09_Error_PDF_Data.xlsx')
    with pd.ExcelWriter(f09, engine='openpyxl') as w:
        x_pdf = np.linspace(0, 50, 500)
        pdf_dict = {'Error_Percent': x_pdf}
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            ef = s['relative_errors'][s['relative_errors'] <= 50]
            try:
                kde = gaussian_kde(ef, bw_method='scott')
                pdf_dict[f'{lbl}_PDF'] = kde(x_pdf)
            except Exception:
                pdf_dict[f'{lbl}_PDF'] = np.zeros(len(x_pdf))
        pd.DataFrame(pdf_dict).to_excel(w, sheet_name='PDF_KDE', index=False)
        stats_rows = []
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            rel = s['relative_errors']
            stats_rows.append({
                'Dataset': lbl, 'Mean': rel.mean(), 'Median': np.median(rel),
                'Std': rel.std(), 'Skewness': float(pd.Series(rel).skew()),
                'Kurtosis': float(pd.Series(rel).kurt()),
                'Mode_approx': x_pdf[np.argmax(gaussian_kde(
                    rel[rel <= 50], bw_method='scott')(x_pdf))],
            })
        pd.DataFrame(stats_rows).to_excel(
            w, sheet_name='PDF_Statistics', index=False)
    print(f"    ✓ 09_Error_PDF_Data.xlsx")

    f10 = os.path.join(excel_dir, '10_Cumulative_Error_Data.xlsx')
    with pd.ExcelWriter(f10, engine='openpyxl') as w:
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            ae_sorted = np.sort(s['absolute_errors'])
            cum_err   = np.cumsum(ae_sorted)
            total_err = cum_err[-1]
            n         = len(ae_sorted)
            pd.DataFrame({
                'Sample_Rank_Pct':        np.arange(1, n+1) / n * 100,
                'Abs_Error_Sorted':       ae_sorted,
                'Cumulative_Error':       cum_err,
                'Cumulative_Error_Pct':   cum_err / total_err * 100,
            }).to_excel(w, sheet_name=f'CumError_{lbl}', index=False)
    print(f"    ✓ 10_Cumulative_Error_Data.xlsx")

    f11 = os.path.join(excel_dir, '11_Error_Quantile_Data.xlsx')
    with pd.ExcelWriter(f11, engine='openpyxl') as w:
        for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
            rel_sorted = np.sort(s['relative_errors'])
            n          = len(rel_sorted)
            probs      = np.arange(1, n+1) / (n+1)
            lambda_hat = 1.0 / rel_sorted.mean() if rel_sorted.mean() > 0 else 1.0
            theo_q_exp = stats.expon.ppf(probs, scale=1/lambda_hat)
            res_std = (rel_sorted - rel_sorted.mean()) / rel_sorted.std()
            theo_q_norm = stats.norm.ppf(probs)
            pd.DataFrame({
                'Empirical_Quantile':      rel_sorted,
                'Theoretical_Q_Expon':     theo_q_exp,
                'Theoretical_Q_Normal':    theo_q_norm,
                'Standardized_Error':      res_std,
                'Probability':             probs,
            }).to_excel(w, sheet_name=f'Quantile_{lbl}', index=False)
    print(f"    ✓ 11_Error_Quantile_Data.xlsx")

    f00 = os.path.join(excel_dir, '00_Origin_Plotting_Guide.xlsx')
    with pd.ExcelWriter(f00, engine='openpyxl') as w:
        pd.DataFrame({
            'File': [
                '01_CDF_Data.xlsx', '02_Radar_Chart_Data.xlsx',
                '03_Error_Histograms.xlsx', '04_Prediction_Scatter.xlsx',
                '05_GP_Uncertainty.xlsx', '06_Threshold_Bar_Data.xlsx',
                '07_Boxplot_Data.xlsx', '08_Exceedance_Data.xlsx',
                '09_Error_PDF_Data.xlsx', '10_Cumulative_Error_Data.xlsx',
                '11_Error_Quantile_Data.xlsx',
            ],
            'Corresponding_Figure': [
                '01_CDF_All_Train_Test.png', '08_Radar_Chart.png',
                '07_Error_Histograms.png', '02_Prediction_Scatter.png',
                '(removed)', '09_Threshold_Bar_Chart.png',
                '10_Error_Boxplot.png', '03_Exceedance_Probability.png',
                '04_Error_PDF.png', '05_Cumulative_Error_Contribution.png',
                '06_Error_Quantile_Plot.png',
            ],
            'Chart_Type': [
                'Line+CI Band', 'Radar/Spider', 'Histogram', 'Scatter',
                'Scatter+ErrorBar', 'Grouped Bar', 'Box Plot',
                'Line+CI Band', 'Line/KDE', 'Line', 'Scatter',
            ],
        }).to_excel(w, sheet_name='Plotting_Guide', index=False)
    print(f"    ✓ 00_Origin_Plotting_Guide.xlsx")
    print(f"    ✅ 所有Origin Excel文件已生成！")


def save_data_files(bundle, sa, st, se, out_dir):
    print("\n=== 保存数据文件 ===")
    data_dir  = os.path.join(out_dir, 'data')
    excel_dir = os.path.join(out_dir, 'excel_for_origin')
    mn        = bundle['model_name']

    rows = []
    for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
        rows.append({
            'Dataset': lbl, 'Model': mn, 'N': len(s['y_true']),
            'R2': s['r2'], 'RMSE': s['rmse'], 'MAE': s['mae'],
            'MRE': s['mean_rel_err'], 'MedRE': s['median_rel_err'],
            'F5': s['f5'], 'F10': s['f10'], 'F15': s['f15'],
            'F20': s['f20'], 'GP_Cov95': s['cov95'],
        })
    pd.DataFrame(rows).to_csv(
        os.path.join(data_dir, 'model_summary_with_errors.csv'),
        index=False, encoding='utf-8')
    for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
        pd.DataFrame({
            'Sample_Index':       range(len(s['relative_errors'])),
            'True_Value':         s['y_true'],
            'Predicted_Value':    s['y_pred'],
            'Residual':           s['residuals'],
            'Absolute_Error':     s['absolute_errors'],
            'Relative_Error_Pct': s['relative_errors'],
            'Prediction_Std':     s['y_std'],
        }).to_csv(os.path.join(
            data_dir, 'individual_errors', f'{mn}_{lbl}_errors.csv'),
            index=False)
    print(f"✓ CSV数据已保存")
    _save_excel_for_origin(bundle, sa, st, se, excel_dir, mn)
    return pd.DataFrame(rows)


# ====================== 综合CDF图 ======================
def create_comprehensive_cdf_plot(bundle, sa, st, se, out_dir):
    print("\n=== 创建综合CDF分析图 ===")
    mn     = bundle['model_name']
    MAX_E  = CONFIG['max_error']
    x_grid = np.linspace(0, MAX_E, 1000)
    N_BOOT = CONFIG['bootstrap_n']

    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    (ax1, ax2), (ax3, ax4) = axes

    # 子图1：CDF
    for lbl, s, ls in [('All Data', sa, '-'), ('Train', st, '--'), ('Test', se, ':')]:
        clr   = COLORS.get(lbl.split()[0], '#1f77b4')
        label = f'{lbl}  |  F10={s["f10"]:.2f}  F20={s["f20"]:.2f}'
        _plot_cdf_with_ci(ax1, s['relative_errors'], x_grid, clr, label, ls=ls, n_boot=N_BOOT)
    for thr in [5, 10, 20]:
        ax1.axvline(thr, color='gray', linestyle='--', alpha=0.5, linewidth=1.2)
        ax1.text(thr+0.3, 0.97, f'{thr}%', color='gray', fontsize=9, va='top')
    ax1.set_xlabel('Relative Error (%)',   fontsize=12, fontweight='bold')
    ax1.set_ylabel('Cumulative Frequency', fontsize=12, fontweight='bold')
    ax1.set_title(f'CDF of Prediction Errors (95% Bootstrap CI)\n{mn}',
                  fontsize=12, fontweight='bold')
    ax1.set_xlim(0, MAX_E); ax1.set_ylim(0, 1)
    ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

    # 子图2：反向CDF
    for lbl, s, ls in [('All Data', sa, '-'), ('Train', st, '--'), ('Test', se, ':')]:
        clr = COLORS.get(lbl.split()[0], '#1f77b4')
        ef  = s['relative_errors'][s['relative_errors'] <= MAX_E]
        es  = np.sort(ef)
        cum = np.arange(1, len(es)+1) / len(es)
        yi  = np.interp(x_grid, es, cum, left=0, right=1)
        ax2.plot(x_grid, (1-yi)*100, color=clr, linewidth=2.5,
                 linestyle=ls, label=f'{lbl}  F10-Exceed={(1-s["f10"])*100:.1f}%', zorder=4)
        lo, hi = bootstrap_cdf(ef, x_grid, N_BOOT)
        ax2.fill_between(x_grid, (1-hi)*100, (1-lo)*100, color=clr, alpha=0.15)
    for thr in [5, 10, 20]:
        ax2.axvline(thr, color='gray', linestyle='--', alpha=0.5, linewidth=1.2)
    ax2.set_xlabel('Relative Error (%)',          fontsize=12, fontweight='bold')
    ax2.set_ylabel('Exceedance Probability (%)',  fontsize=12, fontweight='bold')
    ax2.set_title('Exceedance Probability (1-CDF)', fontsize=12, fontweight='bold')
    ax2.set_xlim(0, MAX_E); ax2.set_ylim(0, 100)
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    # 子图3：PDF
    x_pdf = np.linspace(0, 40, 500)
    for lbl, s, ls in [('All Data', sa, '-'), ('Train', st, '--'), ('Test', se, ':')]:
        clr = COLORS.get(lbl.split()[0], '#1f77b4')
        ef  = s['relative_errors'][s['relative_errors'] <= 50]
        try:
            kde = gaussian_kde(ef, bw_method='scott')
            ax3.plot(x_pdf, kde(x_pdf), color=clr, linewidth=2.5,
                     linestyle=ls, label=f'{lbl}  Mean={s["mean_rel_err"]:.2f}%')
            ax3.fill_between(x_pdf, 0, kde(x_pdf), color=clr, alpha=0.10)
        except Exception:
            pass
    ax3.set_xlabel('Relative Error (%)',         fontsize=12, fontweight='bold')
    ax3.set_ylabel('Probability Density',        fontsize=12, fontweight='bold')
    ax3.set_title('Error PDF (KDE Smoothed)',     fontsize=12, fontweight='bold')
    ax3.set_xlim(0, 40); ax3.set_ylim(bottom=0)
    ax3.legend(fontsize=9); ax3.grid(alpha=0.3)

    # 子图4：散点图
    ax4.scatter(st['y_true'], st['y_pred'], alpha=0.6, color='#2ca02c',
                s=45, edgecolors='none', label=f'Train  R²={st["r2"]:.4f}')
    ax4.scatter(se['y_true'], se['y_pred'], alpha=0.85, color='#d62728',
                s=60, edgecolors='white', linewidths=0.4,
                label=f'Test   R²={se["r2"]:.4f}')
    mn_v = min(sa['y_true'].min(), sa['y_pred'].min())
    mx_v = max(sa['y_true'].max(), sa['y_pred'].max())
    ax4.plot([mn_v, mx_v], [mn_v, mx_v], 'k--', linewidth=2, label='Perfect')
    ax4.set_xlabel('True KQ (MPa·m¹/²)',      fontsize=12, fontweight='bold')
    ax4.set_ylabel('Predicted KQ (MPa·m¹/²)', fontsize=12, fontweight='bold')
    ax4.set_title('Predicted vs True Values',  fontsize=12, fontweight='bold')
    ax4.legend(fontsize=10); ax4.grid(alpha=0.3)

    fig.suptitle(f'TOP1 Model CDF Analysis  |  {mn}\n'
                 f'Train R²={st["r2"]:.4f}   Test R²={se["r2"]:.4f}   '
                 f'Test F10={se["f10"]*100:.1f}%   Test F20={se["f20"]*100:.1f}%',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'figures', 'comprehensive_cdf_analysis.png'),
                dpi=CONFIG['figure_dpi'], bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✓ 综合CDF图已保存")


# ============================================================
#   单独图生成 v5（全部增强注释）
# ============================================================
def create_individual_model_plots(bundle, sa, st, se, out_dir):
    print("\n=== 创建单独分析图（共10张）v5增强版 ===")
    mn      = bundle['model_name']
    ind_dir = os.path.join(out_dir, 'figures', 'individual_models')
    MAX_E   = CONFIG['max_error']
    x_grid  = np.linspace(0, MAX_E, 1000)
    N_BOOT  = CONFIG['bootstrap_n']
    thr_list = CONFIG['error_thresholds']

    # ================================================================
    # 图1：CDF曲线  ★ 增强：分区着色、关键百分位标注、统计文本框
    # ================================================================
    fig, ax = plt.subplots(figsize=(11, 7))

    # 背景分区着色（优秀/良好/可接受）
    ax.axvspan(0,  5,  alpha=0.06, color='green',  zorder=0)
    ax.axvspan(5,  10, alpha=0.06, color='limegreen', zorder=0)
    ax.axvspan(10, 20, alpha=0.06, color='orange', zorder=0)
    ax.axvspan(20, MAX_E, alpha=0.04, color='red', zorder=0)
    # 分区标签
    for x0, x1, lbl_z, clr_z in [
        (0, 5, 'Excellent', 'green'),
        (5, 10, 'Good', 'olive'),
        (10, 20, 'Acceptable', 'darkorange'),
        (20, MAX_E, 'Poor', 'red'),
    ]:
        ax.text((x0+x1)/2, 0.015, lbl_z, ha='center', va='bottom',
                color=clr_z, fontsize=8.5, alpha=0.7, style='italic')

    for lbl, s, ls in [('All Data', sa, '-'), ('Train', st, '--'), ('Test', se, ':')]:
        clr   = COLORS.get(lbl.split()[0], '#1f77b4')
        label = (f'{lbl}  (F5={s["f5"]*100:.1f}%, F10={s["f10"]*100:.1f}%, '
                 f'F15={s["f15"]*100:.1f}%, F20={s["f20"]*100:.1f}%)')
        _plot_cdf_with_ci(ax, s['relative_errors'], x_grid,
                          clr, label, ls=ls, lw=3, n_boot=N_BOOT)
        # 标注各曲线在x=10处的CDF值
        ef  = s['relative_errors'][s['relative_errors'] <= MAX_E]
        es  = np.sort(ef)
        cum = np.arange(1, len(es)+1) / len(es)
        cdf_at_10 = np.interp(10, es, cum)
        ax.annotate(f'{cdf_at_10*100:.1f}%',
                    xy=(10, cdf_at_10),
                    xytext=(10.5, cdf_at_10 - 0.04),
                    fontsize=8.5, color=clr,
                    arrowprops=dict(arrowstyle='->', color=clr, lw=1.2))

    for thr in [5, 10, 20]:
        ax.axvline(thr, color='gray', linestyle='--', alpha=0.6, linewidth=1.5)
        ax.text(thr+0.2, 0.02, f'{thr}%', color='gray', fontsize=10)
    # 水平参考线
    for yv in [0.5, 0.8, 0.9]:
        ax.axhline(yv, color='lightgray', ls=':', lw=1, alpha=0.8)
        ax.text(MAX_E-0.5, yv+0.005, f'{yv*100:.0f}th pct',
                ha='right', fontsize=8, color='gray')

    # 统计文本框
    txt = (f"Test Statistics\n"
           f"N={len(se['y_true'])}  R²={se['r2']:.4f}\n"
           f"RMSE={se['rmse']:.3f}  MAE={se['mae']:.3f}\n"
           f"MRE={se['mean_rel_err']:.2f}%  σ={se['std_rel_err']:.2f}%")
    add_stats_textbox(ax, txt, x=0.97, y=0.05)

    ax.set_xlabel('Relative Error (%)',   fontsize=13, fontweight='bold')
    ax.set_ylabel('Cumulative Frequency', fontsize=13, fontweight='bold')
    ax.set_title(f'CDF of Prediction Errors  (shaded = {int(CONFIG["bootstrap_ci"]*100)}% Bootstrap CI)\n{mn}',
                 fontsize=13, fontweight='bold')
    ax.set_xlim(0, MAX_E); ax.set_ylim(0, 1)
    ax.legend(fontsize=9, loc='lower right'); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(ind_dir, '01_CDF_All_Train_Test.png'),
                dpi=CONFIG['figure_dpi'], bbox_inches='tight', facecolor='white')
    plt.close()
    print("  ✓ 01_CDF_All_Train_Test.png")

    # ================================================================
    # 图2：散点图  ★ 增强：±10%/±20%误差带、密度着色、统计文本框
    # ================================================================
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax_i, (lbl, s, clr) in enumerate(
            [('Train', st, '#2ca02c'), ('Test', se, '#d62728')]):
        ax = axes[ax_i]
        mn_v = min(s['y_true'].min(), s['y_pred'].min()) * 0.95
        mx_v = max(s['y_true'].max(), s['y_pred'].max()) * 1.05
        ref  = np.array([mn_v, mx_v])

        # 误差带（±10% 和 ±20%）
        ax.fill_between(ref, ref*0.8, ref*1.2, alpha=0.08,
                        color='red',    label='±20% band')
        ax.fill_between(ref, ref*0.9, ref*1.1, alpha=0.12,
                        color='orange', label='±10% band')
        ax.fill_between(ref, ref*0.95, ref*1.05, alpha=0.18,
                        color='green',  label='±5% band')

        # 散点（按误差大小着色）
        rel = s['relative_errors']
        sc_plot = ax.scatter(s['y_true'], s['y_pred'],
                             c=rel, cmap='RdYlGn_r',
                             vmin=0, vmax=30,
                             alpha=0.80, s=55,
                             edgecolors='black', linewidths=0.3, zorder=5)
        plt.colorbar(sc_plot, ax=ax, label='Relative Error (%)', fraction=0.046)

        ax.plot(ref, ref, 'k-', lw=2.5, label='Perfect Prediction', zorder=6)

        # 标注离群点（误差>20%）
        high_err_mask = rel > 20
        if high_err_mask.sum() > 0:
            for xt, xp, er in zip(s['y_true'][high_err_mask],
                                   s['y_pred'][high_err_mask],
                                   rel[high_err_mask]):
                ax.annotate(f'{er:.1f}%',
                            xy=(xt, xp), xytext=(xt+0.3, xp+0.3),
                            fontsize=7.5, color='darkred',
                            arrowprops=dict(arrowstyle='->', color='darkred', lw=0.8))

        # 统计文本框
        txt = (f"N={len(s['y_true'])}\n"
               f"R²={s['r2']:.4f}\n"
               f"RMSE={s['rmse']:.3f}\n"
               f"MAE={s['mae']:.3f}\n"
               f"MRE={s['mean_rel_err']:.2f}%\n"
               f"F10={s['f10']*100:.1f}%  F20={s['f20']*100:.1f}%\n"
               f"Outliers(>20%)={high_err_mask.sum()}")
        add_stats_textbox(ax, txt, x=0.03, y=0.97, ha='left', va='top')

        ax.set_xlim(mn_v, mx_v); ax.set_ylim(mn_v, mx_v)
        ax.set_xlabel('True KQ (MPa·m¹/²)',      fontsize=12, fontweight='bold')
        ax.set_ylabel('Predicted KQ (MPa·m¹/²)', fontsize=12, fontweight='bold')
        ax.set_title(f'{lbl} Set  —  Predicted vs True\n'
                     f'(color = relative error, bands = ±5/10/20%)',
                     fontsize=12, fontweight='bold')
        ax.legend(fontsize=9, loc='lower right'); ax.grid(alpha=0.3)
        ax.set_aspect('equal', adjustable='box')

    plt.suptitle(f'Prediction Scatter Plot  |  {mn}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(ind_dir, '02_Prediction_Scatter.png'),
                dpi=CONFIG['figure_dpi'], bbox_inches='tight', facecolor='white')
    plt.close()
    print("  ✓ 02_Prediction_Scatter.png")

    # ================================================================
    # 图3：反向CDF  ★ 增强：达标/警戒/危险区分区、关键点标注
    # ================================================================
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    ax = axes[0]
    # 背景分区：从上到下危险→警戒→达标
    ax.axhspan(50, 100, alpha=0.06, color='red')
    ax.axhspan(20,  50, alpha=0.06, color='orange')
    ax.axhspan(10,  20, alpha=0.06, color='yellow')
    ax.axhspan( 0,  10, alpha=0.06, color='green')
    # 分区标签
    for yb, yt, lbl_z, clr_z in [
        (50, 100, 'Danger Zone', 'red'),
        (20,  50, 'Caution Zone', 'darkorange'),
        (10,  20, 'Warning Zone', 'goldenrod'),
        ( 0,  10, 'Safe Zone', 'green'),
    ]:
        ax.text(MAX_E*0.99, (yb+yt)/2, lbl_z, ha='right', va='center',
                color=clr_z, fontsize=8.5, alpha=0.75, style='italic')

    for lbl, s, ls in [('All Data', sa, '-'), ('Train', st, '--'), ('Test', se, ':')]:
        clr = COLORS.get(lbl.split()[0], '#1f77b4')
        ef  = s['relative_errors'][s['relative_errors'] <= MAX_E]
        es  = np.sort(ef)
        cum = np.arange(1, len(es)+1) / len(es)
        yi  = np.interp(x_grid, es, cum, left=0, right=1)
        exceed = (1 - yi) * 100
        ax.plot(x_grid, exceed, color=clr, linewidth=3,
                linestyle=ls, zorder=4,
                label=f'{lbl}  (@10%={(1-s["f10"])*100:.1f}%,'
                      f' @20%={(1-s["f20"])*100:.1f}%)')
        lo, hi = bootstrap_cdf(ef, x_grid, N_BOOT)
        ax.fill_between(x_grid, (1-hi)*100, (1-lo)*100,
                        color=clr, alpha=0.15, zorder=3)
        # 标注关键阈值交叉点
        for thr_key in [10, 20]:
            exceed_at = (1 - np.interp(thr_key, es, cum)) * 100
            ax.plot(thr_key, exceed_at, 'o', color=clr, ms=8,
                    markeredgecolor='black', markeredgewidth=0.8, zorder=6)
            ax.annotate(f'{exceed_at:.1f}%',
                        xy=(thr_key, exceed_at),
                        xytext=(thr_key+1.0, exceed_at+2),
                        fontsize=8, color=clr,
                        arrowprops=dict(arrowstyle='->', color=clr, lw=1.0))

    for thr, clr_r in zip([5, 10, 20], ['#e6194b', '#f58231', '#808080']):
        ax.axvline(thr, color=clr_r, linestyle='--', alpha=0.6, linewidth=1.5)
        ax.text(thr+0.3, 96, f'{thr}%\nthreshold',
                color=clr_r, fontsize=8.5, va='top', ha='left')
    # 参考水平线
    for yref, lbl_r in [(10, '10% accept'), (5, '5% ideal')]:
        ax.axhline(yref, color='navy', ls=':', lw=1.5, alpha=0.7)
        ax.text(0.5, yref+0.8, lbl_r, fontsize=8.5, color='navy', alpha=0.8)

    ax.set_xlabel('Relative Error Threshold (%)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Exceedance Probability (%)',   fontsize=13, fontweight='bold')
    ax.set_title('Exceedance Probability Plot (1 − CDF)\n'
                 '(% of samples with error > threshold)',
                 fontsize=13, fontweight='bold')
    ax.set_xlim(0, MAX_E); ax.set_ylim(0, 100)
    ax.legend(fontsize=9, loc='upper right'); ax.grid(alpha=0.3)

    # 右图：关键阈值下超标率柱状图（与v4相同，增加颜色编码）
    ax = axes[1]
    key_thrs = [5, 10, 15, 20, 25, 30]
    x_pos    = np.arange(len(key_thrs))
    width    = 0.28
    exceed_tr = [(1 - st[f'f{t}']) * 100 for t in key_thrs]
    exceed_te = [(1 - se[f'f{t}']) * 100 for t in key_thrs]
    exceed_al = [(1 - sa[f'f{t}']) * 100 for t in key_thrs]

    # 背景色按达标情况
    for i, thr_v in enumerate(key_thrs):
        bg_clr = 'lightgreen' if exceed_te[i] <= 10 else (
                  'lightyellow' if exceed_te[i] <= 20 else 'mistyrose')
        ax.axvspan(i - 0.5, i + 0.5, alpha=0.25, color=bg_clr, zorder=0)

    b1 = ax.bar(x_pos - width, exceed_tr, width, color='#2ca02c',
                alpha=0.8, edgecolor='black', lw=0.8, label='Train')
    b2 = ax.bar(x_pos,         exceed_te, width, color='#d62728',
                alpha=0.8, edgecolor='black', lw=0.8, label='Test')
    b3 = ax.bar(x_pos + width, exceed_al, width, color='#1f77b4',
                alpha=0.8, edgecolor='black', lw=0.8, label='All')

    for bars, vals in [(b1, exceed_tr), (b2, exceed_te), (b3, exceed_al)]:
        for bar, v in zip(bars, vals):
            if v > 0.5:
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.5,
                        f'{v:.1f}%', ha='center', va='bottom',
                        fontsize=8.5, fontweight='bold')

    ax.axhline(10, color='orange', ls='--', lw=1.8, alpha=0.85,
               label='10% acceptable limit')
    ax.axhline(5,  color='green',  ls='--', lw=1.8, alpha=0.85,
               label='5% ideal target')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'F{t}%' for t in key_thrs],
                       fontsize=11, fontweight='bold')
    ax.set_xlabel('Error Threshold',         fontsize=13, fontweight='bold')
    ax.set_ylabel('Exceedance Rate (%)',     fontsize=13, fontweight='bold')
    ax.set_title('Exceedance Rate at Key Thresholds\n(Lower is Better  |  bg = Test performance zone)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, max(max(exceed_te)*1.25, 30))

    fig.suptitle(f'Exceedance Probability Analysis  |  {mn}\n'
                 f'Test: {(1-se["f10"])*100:.1f}% exceed 10%,  '
                 f'{(1-se["f20"])*100:.1f}% exceed 20%',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(ind_dir, '03_Exceedance_Probability.png'),
                dpi=CONFIG['figure_dpi'], bbox_inches='tight', facecolor='white')
    plt.close()
    print("  ✓ 03_Exceedance_Probability.png")

    # ================================================================
    # 图4：误差PDF  ★ 增强：峰值标注、分区着色、IQR范围、统计框
    # ================================================================
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    x_pdf = np.linspace(0, 40, 500)

    # ── 左图：三条PDF曲线 ──────────────────────────────
    ax = axes[0]
    # 背景分区（与图1保持一致）
    ax.axvspan(0,  5,  alpha=0.07, color='green',     zorder=0)
    ax.axvspan(5,  10, alpha=0.07, color='limegreen',  zorder=0)
    ax.axvspan(10, 20, alpha=0.07, color='orange',     zorder=0)
    ax.axvspan(20, 40, alpha=0.05, color='red',        zorder=0)
    for x0, x1, lbl_z, clr_z in [
        (0, 5, 'Excellent', 'green'),
        (5, 10, 'Good', 'olive'),
        (10, 20, 'Acceptable', 'darkorange'),
        (20, 40, 'Poor', 'red'),
    ]:
        ax.text((x0+x1)/2, -0.001, lbl_z, ha='center', va='top',
                color=clr_z, fontsize=8, alpha=0.7, style='italic',
                transform=ax.get_xaxis_transform())

    peak_info_lines = []  # 收集峰值信息用于标注
    for lbl, s, ls in [('All Data', sa, '-'), ('Train', st, '--'), ('Test', se, ':')]:
        clr = COLORS.get(lbl.split()[0], '#1f77b4')
        ef  = s['relative_errors'][s['relative_errors'] <= 50]
        try:
            kde = gaussian_kde(ef, bw_method='scott')
            pdf_vals = kde(x_pdf)
            peak_x = x_pdf[np.argmax(pdf_vals)]
            peak_y = pdf_vals.max()

            ax.plot(x_pdf, pdf_vals, color=clr, linewidth=3,
                    linestyle=ls, zorder=4,
                    label=(f'{lbl}  Mode≈{peak_x:.1f}%  '
                           f'Mean={s["mean_rel_err"]:.1f}%  '
                           f'Median={s["median_rel_err"]:.1f}%'))
            ax.fill_between(x_pdf, 0, pdf_vals, color=clr, alpha=0.10)

            # 峰值标注
            ax.annotate(f'Mode≈{peak_x:.1f}%',
                        xy=(peak_x, peak_y),
                        xytext=(peak_x + 2.5, peak_y * 0.92),
                        fontsize=8.5, color=clr, fontweight='bold',
                        arrowprops=dict(arrowstyle='->', color=clr, lw=1.2))
            ax.plot(peak_x, peak_y, '*', color=clr, ms=10, zorder=7)

            # 均值线（实线点）和中位数线（虚线）
            ax.axvline(s['mean_rel_err'],   color=clr, ls=':', lw=1.8, alpha=0.8)
            ax.axvline(s['median_rel_err'], color=clr, ls='--', lw=1.8, alpha=0.8)

            # IQR 阴影
            q25 = np.percentile(ef, 25)
            q75 = np.percentile(ef, 75)
            mask = (x_pdf >= q25) & (x_pdf <= q75)
            ax.fill_between(x_pdf[mask], 0, pdf_vals[mask],
                            color=clr, alpha=0.22, zorder=3,
                            label=f'_{lbl} IQR [{q25:.1f}–{q75:.1f}%]')

            peak_info_lines.append(
                f"{lbl}: Mode={peak_x:.1f}%  Skew={float(pd.Series(ef).skew()):.2f}  "
                f"Kurt={float(pd.Series(ef).kurt()):.2f}")
        except Exception:
            pass

    # 竖线图例说明
    ax.plot([], [], 'k:', lw=1.8, label='Dotted = Mean')
    ax.plot([], [], 'k--', lw=1.8, label='Dashed = Median')

    ax.set_xlabel('Relative Error (%)',  fontsize=13, fontweight='bold')
    ax.set_ylabel('Probability Density', fontsize=13, fontweight='bold')
    ax.set_title('Error PDF  (KDE Smoothed)\n'
                 '★=Mode  dotted=Mean  dashed=Median  shaded=IQR',
                 fontsize=13, fontweight='bold')
    ax.set_xlim(0, 40); ax.set_ylim(bottom=0)
    ax.legend(fontsize=8.5, loc='upper right'); ax.grid(alpha=0.3)

    # 统计文本框（左图右下角）
    txt_pdf = "\n".join(peak_info_lines)
    add_stats_textbox(ax, txt_pdf, x=0.97, y=0.97, va='top', fontsize=8)

    # ── 右图：Train vs Test 差异着色 ──────────────────
    ax = axes[1]
    ef_tr = st['relative_errors'][st['relative_errors'] <= 40]
    ef_te = se['relative_errors'][se['relative_errors'] <= 40]
    try:
        kde_tr = gaussian_kde(ef_tr, bw_method='scott')
        kde_te = gaussian_kde(ef_te, bw_method='scott')
        pdf_tr = kde_tr(x_pdf)
        pdf_te = kde_te(x_pdf)

        ax.plot(x_pdf, pdf_tr, color='#2ca02c', lw=3,
                label=(f'Train  Skew={float(pd.Series(ef_tr).skew()):.2f}  '
                       f'Kurt={float(pd.Series(ef_tr).kurt()):.2f}'))
        ax.plot(x_pdf, pdf_te, color='#d62728', lw=3, ls='--',
                label=(f'Test   Skew={float(pd.Series(ef_te).skew()):.2f}  '
                       f'Kurt={float(pd.Series(ef_te).kurt()):.2f}'))
        ax.fill_between(x_pdf, pdf_tr, pdf_te,
                        where=pdf_tr > pdf_te, alpha=0.28,
                        color='#2ca02c', label='Train > Test (Train better here)')
        ax.fill_between(x_pdf, pdf_tr, pdf_te,
                        where=pdf_te > pdf_tr, alpha=0.28,
                        color='#d62728', label='Test > Train (Test worse here)')

        # 分布交叉点标注
        diff = pdf_tr - pdf_te
        sign_changes = np.where(np.diff(np.sign(diff)))[0]
        for sc_i in sign_changes:
            x_cross = (x_pdf[sc_i] + x_pdf[sc_i+1]) / 2
            y_cross = (pdf_tr[sc_i] + pdf_te[sc_i]) / 2
            ax.plot(x_cross, y_cross, 'kD', ms=6, zorder=7)
            ax.annotate(f'Cross\n{x_cross:.1f}%',
                        xy=(x_cross, y_cross),
                        xytext=(x_cross + 1.5, y_cross * 1.15),
                        fontsize=8, color='black',
                        arrowprops=dict(arrowstyle='->', color='black', lw=1))

        # 峰值标注
        for pdf_vals, ef_vals, clr_v in [
            (pdf_tr, ef_tr, '#2ca02c'), (pdf_te, ef_te, '#d62728')
        ]:
            pk_x = x_pdf[np.argmax(pdf_vals)]
            pk_y = pdf_vals.max()
            ax.plot(pk_x, pk_y, '*', color=clr_v, ms=11, zorder=8)
            ax.annotate(f'{pk_x:.1f}%', xy=(pk_x, pk_y),
                        xytext=(pk_x + 1.5, pk_y * 0.95),
                        fontsize=8.5, color=clr_v, fontweight='bold',
                        arrowprops=dict(arrowstyle='->', color=clr_v, lw=1))

        # KS统计量
        ks_stat, ks_p = stats.ks_2samp(ef_tr, ef_te)
        txt_ks = (f"KS Test\nD={ks_stat:.3f}\np={ks_p:.3f}\n"
                  f"{'≈ Same dist.' if ks_p > 0.05 else 'Different dist.'}")
        add_stats_textbox(ax, txt_ks, x=0.97, y=0.97, va='top', fontsize=9)

    except Exception:
        pass

    ax.set_xlabel('Relative Error (%)',  fontsize=13, fontweight='bold')
    ax.set_ylabel('Probability Density', fontsize=13, fontweight='bold')
    ax.set_title('Train vs Test PDF Comparison\n'
                 '(★=Mode, ◆=Cross-over point, shading=difference)',
                 fontsize=13, fontweight='bold')
    ax.set_xlim(0, 40); ax.set_ylim(bottom=0)
    ax.legend(fontsize=9, loc='upper right'); ax.grid(alpha=0.3)

    fig.suptitle(f'Error PDF Analysis  |  {mn}\n'
                 f'Train Mean={st["mean_rel_err"]:.2f}%  '
                 f'Test Mean={se["mean_rel_err"]:.2f}%  '
                 f'Gap={se["mean_rel_err"]-st["mean_rel_err"]:.2f}%',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(ind_dir, '04_Error_PDF.png'),
                dpi=CONFIG['figure_dpi'], bbox_inches='tight', facecolor='white')
    plt.close()
    print("  ✓ 04_Error_PDF.png")

    # ================================================================
    # 图5：累积误差贡献  ★ 增强：基尼系数、关键点标注、帕累托线
    # ================================================================
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    ax = axes[0]
    for lbl, s, ls in [('All Data', sa, '-'), ('Train', st, '--'), ('Test', se, ':')]:
        clr      = COLORS.get(lbl.split()[0], '#1f77b4')
        ae_sorted = np.sort(s['absolute_errors'])
        cum_err   = np.cumsum(ae_sorted)
        total_err = cum_err[-1]
        n         = len(ae_sorted)
        x_pct     = np.arange(1, n+1) / n * 100
        y_pct     = cum_err / total_err * 100

        # 基尼系数
        gini = compute_gini(s['absolute_errors'])

        ax.plot(x_pct, y_pct, color=clr, lw=3, linestyle=ls, zorder=4,
                label=f'{lbl}  Gini={gini:.3f}')

        # 标注关键点：20%, 50%, 80%
        for pct_mark in [20, 50, 80]:
            idx_m = np.searchsorted(x_pct, pct_mark)
            if idx_m < len(y_pct):
                ym = y_pct[idx_m]
                ax.plot(pct_mark, ym, 'o', color=clr, ms=7,
                        markeredgecolor='black', markeredgewidth=0.7, zorder=6)
                ax.annotate(f'{ym:.1f}%',
                            xy=(pct_mark, ym),
                            xytext=(pct_mark + 2.5, ym - 4),
                            fontsize=8, color=clr,
                            arrowprops=dict(arrowstyle='->', color=clr, lw=0.9))

    # 帕累托80/20参考线
    ax.axvline(20, color='navy', ls=':', lw=1.5, alpha=0.7)
    ax.axhline(80, color='navy', ls=':', lw=1.5, alpha=0.7, label='Pareto 80/20 ref')
    ax.plot([0, 100], [0, 100], 'k--', lw=1.5, alpha=0.5,
            label='Uniform distribution')
    # 帕累托区域着色
    ax.fill_between([0, 20], [0, 0], [0, 80], alpha=0.04, color='blue')
    ax.text(10, 40, 'Pareto\n80/20', ha='center', fontsize=9,
            color='navy', alpha=0.6, style='italic')

    ax.set_xlabel('Sample Rank (% from smallest error)',
                  fontsize=13, fontweight='bold')
    ax.set_ylabel('Cumulative Error Contribution (%)',
                  fontsize=13, fontweight='bold')
    ax.set_title('Cumulative Error Contribution  (Lorenz Curve)\n'
                 '(Gini = 0 → uniform distribution; Gini → 1 → concentrated)',
                 fontsize=13, fontweight='bold')
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    ax.legend(fontsize=9, loc='upper left'); ax.grid(alpha=0.3)

    # 右图：误差集中度分析
    ax = axes[1]
    pct_groups = [10, 20, 30, 50, 80]
    conc_data  = {}
    for lbl, s in [('Train', st), ('Test', se), ('All', sa)]:
        ae_sorted = np.sort(s['absolute_errors'])
        cum_err   = np.cumsum(ae_sorted)
        total_err = cum_err[-1]
        n         = len(ae_sorted)
        x_pct     = np.arange(1, n+1) / n * 100
        y_pct     = cum_err / total_err * 100
        conc_data[lbl] = [np.interp(p, x_pct, y_pct) for p in pct_groups]

    x_pos = np.arange(len(pct_groups))
    width = 0.28
    for i, (lbl, clr) in enumerate([('Train', '#2ca02c'),
                                     ('Test',  '#d62728'),
                                     ('All',   '#1f77b4')]):
        vals = conc_data[lbl]
        bars = ax.bar(x_pos + (i-1)*width, vals, width,
                      color=clr, alpha=0.80, edgecolor='black', lw=0.8,
                      label=lbl)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.5,
                    f'{v:.1f}%', ha='center', va='bottom',
                    fontsize=8.5, fontweight='bold')

    ax.plot(x_pos, pct_groups, 'k--o', lw=2, ms=7, alpha=0.7,
            label='Uniform reference')
    # 差距标注（Test vs Uniform）
    for xi, (pg, tv) in enumerate(zip(pct_groups, conc_data['Test'])):
        gap = pg - tv
        ax.annotate(f'Δ={gap:.1f}%',
                    xy=(xi, tv),
                    xytext=(xi + 0.02, tv + 4),
                    fontsize=8, color='#d62728',
                    arrowprops=dict(arrowstyle='->', color='#d62728', lw=0.8))

    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'Bottom {p}%' for p in pct_groups],
                       fontsize=10, fontweight='bold')
    ax.set_xlabel('Sample Group (ranked by error, small→large)',
                  fontsize=13, fontweight='bold')
    ax.set_ylabel('Cumulative Error Contribution (%)',
                  fontsize=13, fontweight='bold')
    ax.set_title('Error Concentration Analysis\n'
                 '(Δ = deviation from uniform; larger Δ = more concentrated)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 108)

    gini_all   = compute_gini(sa['absolute_errors'])
    gini_train = compute_gini(st['absolute_errors'])
    gini_test  = compute_gini(se['absolute_errors'])
    fig.suptitle(f'Cumulative Error Contribution  |  {mn}\n'
                 f'Gini: All={gini_all:.3f}  Train={gini_train:.3f}  Test={gini_test:.3f}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(ind_dir, '05_Cumulative_Error_Contribution.png'),
                dpi=CONFIG['figure_dpi'], bbox_inches='tight', facecolor='white')
    plt.close()
    print("  ✓ 05_Cumulative_Error_Contribution.png")

    # ================================================================
    # 图6：误差分位数图  ★ 增强：分段着色、R²拟合度、离群点标注
    # ================================================================
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # ── 左图：指数QQ图 ────────────────────────────────
    ax = axes[0]
    all_r2_vals = []
    for lbl, s, clr, ls in [('All',   sa, '#1f77b4', 'o'),
                              ('Train', st, '#2ca02c', 's'),
                              ('Test',  se, '#d62728', '^')]:
        rel_sorted = np.sort(s['relative_errors'])
        n          = len(rel_sorted)
        probs      = np.arange(1, n+1) / (n+1)
        lambda_hat = 1.0 / rel_sorted.mean() if rel_sorted.mean() > 0 else 1.0
        theo_q     = stats.expon.ppf(probs, scale=1/lambda_hat)

        # R² of QQ fit
        qq_r2 = r2_score(theo_q, rel_sorted) if len(theo_q) > 2 else np.nan
        all_r2_vals.append((lbl, qq_r2))

        ax.scatter(theo_q, rel_sorted, color=clr, alpha=0.60, s=35,
                   edgecolors='black', lw=0.3, label=f'{lbl}  QQ-R²={qq_r2:.3f}',
                   marker=ls, zorder=4)

        # 标注严重离群点（偏离对角线超过50%）
        ref_line = theo_q
        deviation = np.abs(rel_sorted - ref_line) / (ref_line + 1e-6)
        outlier_mask = deviation > 0.5
        for tx, ty in zip(theo_q[outlier_mask], rel_sorted[outlier_mask]):
            ax.annotate(f'({tx:.1f},{ty:.1f})',
                        xy=(tx, ty), xytext=(tx + 0.5, ty * 1.05),
                        fontsize=7, color=clr, alpha=0.8)

    all_sorted = np.sort(sa['relative_errors'])
    n_all      = len(all_sorted)
    probs_all  = np.arange(1, n_all+1) / (n_all+1)
    lam_all    = 1.0 / all_sorted.mean()
    theo_all   = stats.expon.ppf(probs_all, scale=1/lam_all)
    ref_max    = max(theo_all.max(), all_sorted.max()) * 1.05
    ax.plot([0, ref_max], [0, ref_max], 'k--', lw=2.2, alpha=0.8,
            label='Perfect fit (y=x)')
    # 上下偏差带（±30%）
    ref_x = np.array([0, ref_max])
    ax.fill_between(ref_x, ref_x*0.7, ref_x*1.3, alpha=0.06,
                    color='blue', label='±30% band')

    ax.set_xlabel('Theoretical Quantiles (Exponential)',
                  fontsize=13, fontweight='bold')
    ax.set_ylabel('Empirical Quantiles (Actual Error %)',
                  fontsize=13, fontweight='bold')
    ax.set_title('Error Q-Q Plot  (vs Exponential Distribution)\n'
                 'Points on diagonal → errors follow exponential dist.',
                 fontsize=13, fontweight='bold')
    ax.set_xlim(0); ax.set_ylim(0)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # ── 右图：正态QQ图 ────────────────────────────────
    ax = axes[1]
    for lbl, s, clr, ls in [('All',   sa, '#1f77b4', 'o'),
                              ('Train', st, '#2ca02c', 's'),
                              ('Test',  se, '#d62728', '^')]:
        rel  = s['relative_errors']
        res_std = (rel - rel.mean()) / rel.std()
        n       = len(res_std)
        probs   = np.arange(1, n+1) / (n+1)
        theo_n  = stats.norm.ppf(probs)
        sorted_std = np.sort(res_std)

        # Shapiro-Wilk 正态检验
        if n <= 5000:
            sw_stat, sw_p = stats.shapiro(res_std[:5000])
        else:
            sw_stat, sw_p = np.nan, np.nan

        ax.scatter(theo_n, sorted_std, color=clr, alpha=0.60, s=35,
                   edgecolors='black', lw=0.3,
                   label=f'{lbl}  SW-p={sw_p:.3f}', marker=ls, zorder=4)

    max_abs = max(3.5,
                  np.abs(np.sort(
                    (sa['relative_errors'] - sa['relative_errors'].mean())
                    / sa['relative_errors'].std()
                  )).max() * 0.95)
    lim = min(max_abs, 4.5)
    ax.plot([-lim, lim], [-lim, lim], 'k--', lw=2.2, alpha=0.8,
            label='Normal fit (y=x)')
    # 分位数分段着色（尾部）
    ax.axvspan( 2,  lim, alpha=0.07, color='red')
    ax.axvspan(-lim, -2, alpha=0.07, color='red')
    ax.axvspan( 1,   2,  alpha=0.05, color='orange')
    ax.axvspan(-2,  -1,  alpha=0.05, color='orange')
    ax.text( lim*0.85,  lim*0.7,  'Upper\ntail', ha='center',
             fontsize=8.5, color='red', alpha=0.7)
    ax.text(-lim*0.85,  lim*0.7,  'Lower\ntail', ha='center',
             fontsize=8.5, color='red', alpha=0.7)
    ax.axhline(0, color='gray', ls=':', lw=1, alpha=0.5)
    ax.axvline(0, color='gray', ls=':', lw=1, alpha=0.5)

    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel('Theoretical Quantiles N(0,1)',
                  fontsize=13, fontweight='bold')
    ax.set_ylabel('Standardized Empirical Quantiles',
                  fontsize=13, fontweight='bold')
    ax.set_title('Standardized Error Q-Q Plot  (vs Normal)\n'
                 'SW-p > 0.05 → normally distributed  |  shaded = heavy tails',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    fig.suptitle(f'Error Quantile Analysis  |  {mn}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(ind_dir, '06_Error_Quantile_Plot.png'),
                dpi=CONFIG['figure_dpi'], bbox_inches='tight', facecolor='white')
    plt.close()
    print("  ✓ 06_Error_Quantile_Plot.png")

    # ================================================================
    # 图7：误差直方图  ★ 增强：正态/指数拟合曲线、分位数线、统计框
    # ================================================================
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for i, (lbl, s, clr) in enumerate(
            [('Train', st, '#2ca02c'), ('Test', se, '#d62728')]):
        ax = axes[i]
        ef = s['relative_errors'][s['relative_errors'] <= 30]

        # 直方图（归一化为密度）
        n_hist, bins, patches = ax.hist(ef, bins=25, alpha=0.65,
                                        color=clr, edgecolor='black',
                                        density=True, label='Error distribution')

        # 分区颜色
        for patch, left_edge in zip(patches, bins[:-1]):
            if left_edge < 5:
                patch.set_facecolor('#2ca02c')
            elif left_edge < 10:
                patch.set_facecolor('#8fce00')
            elif left_edge < 20:
                patch.set_facecolor('#f6b26b')
            else:
                patch.set_facecolor('#e06666')
            patch.set_alpha(0.75)
            patch.set_edgecolor('black')

        # KDE平滑曲线
        x_fit = np.linspace(0, 30, 300)
        try:
            kde_f = gaussian_kde(ef, bw_method='scott')
            ax.plot(x_fit, kde_f(x_fit), color='black', lw=2.5,
                    label='KDE', zorder=5)
        except Exception:
            pass

        # 指数分布拟合
        lam_fit = 1.0 / ef.mean() if ef.mean() > 0 else 1.0
        exp_pdf = lam_fit * np.exp(-lam_fit * x_fit)
        ax.plot(x_fit, exp_pdf, color='navy', lw=2, ls='--',
                label=f'Exponential fit (λ={lam_fit:.3f})', zorder=4)

        # 均值、中位数、Q25/Q75
        mn_e  = np.mean(ef)
        med_e = np.median(ef)
        q25_e = np.percentile(ef, 25)
        q75_e = np.percentile(ef, 75)
        y_max = n_hist.max() * 1.05

        ax.axvline(mn_e,  color='navy', ls='--', lw=2,
                   label=f'Mean: {mn_e:.2f}%')
        ax.axvline(med_e, color='gold', ls='--', lw=2,
                   label=f'Median: {med_e:.2f}%')
        ax.axvline(q25_e, color='purple', ls=':', lw=1.5,
                   label=f'Q25: {q25_e:.2f}%')
        ax.axvline(q75_e, color='purple', ls=':', lw=1.5,
                   label=f'Q75: {q75_e:.2f}%')
        ax.axvspan(q25_e, q75_e, alpha=0.10, color='purple',
                   label=f'IQR: {q75_e-q25_e:.2f}%')

        # 分区图例标记
        ax.axvspan(0,  5,  alpha=0.0, color='green')
        for x0, x1, lbl_z, clr_z in [
            (0, 5, 'Exc.', 'green'), (5, 10, 'Good', 'olive'),
            (10, 20, 'Acc.', 'darkorange'), (20, 30, 'Poor', 'red'),
        ]:
            ax.text((x0+x1)/2, ax.get_ylim()[1]*0.01 if ax.get_ylim()[1] > 0 else 0.001,
                    lbl_z, ha='center', va='bottom', fontsize=7.5,
                    color=clr_z, alpha=0.7, style='italic',
                    transform=ax.get_xaxis_transform())

        # 统计文本框
        sw_stat2, sw_p2 = stats.shapiro(ef[:5000] if len(ef) > 5000 else ef)
        txt_h = (f"N={len(ef)}\n"
                 f"Mean={mn_e:.2f}%\n"
                 f"Std={ef.std():.2f}%\n"
                 f"Skew={float(pd.Series(ef).skew()):.3f}\n"
                 f"Kurt={float(pd.Series(ef).kurt()):.3f}\n"
                 f"SW-p={sw_p2:.3f}")
        add_stats_textbox(ax, txt_h, x=0.97, y=0.97, va='top', fontsize=8.5)

        ax.set_xlabel('Relative Error (%)',  fontsize=12, fontweight='bold')
        ax.set_ylabel('Probability Density', fontsize=12, fontweight='bold')
        ax.set_title(f'{lbl} Error Distribution\n'
                     f'(Color zones: green=Excellent, yellow-green=Good, '
                     f'orange=Acceptable, red=Poor)',
                     fontsize=11, fontweight='bold')
        ax.legend(fontsize=8.5, loc='upper right'); ax.grid(alpha=0.3)

    plt.suptitle(f'Error Distribution (Histogram + KDE + Fits)  |  {mn}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(ind_dir, '07_Error_Histograms.png'),
                dpi=CONFIG['figure_dpi'], bbox_inches='tight', facecolor='white')
    plt.close()
    print("  ✓ 07_Error_Histograms.png")

    # ================================================================
    # 图8：雷达图  ★ 增强：数值标签、得分等级背景、差距对比填充
    # ================================================================
    metrics = ['R²', 'F5%', 'F10%', 'F15%', 'F20%', 'GP\nCov95%']
    N       = len(metrics)
    angles  = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, axes = plt.subplots(1, 2, figsize=(16, 8),
                              subplot_kw=dict(projection='polar'))

    # ── 左图：三组对比（All/Train/Test）──────────────────────
    ax = axes[0]
    # 等级背景圆环
    levels = [(1.0, 'Excellent', '#d4efdf'), (0.8, 'Good', '#fdebd0'),
              (0.6, 'Fair', '#fce4d6'), (0.4, 'Poor', '#fcd5d5')]
    for rval, lbl_g, bg_c in levels:
        ax.fill(angles, [rval]*len(angles), alpha=0.10, color=bg_c)
    ax.set_yticks([0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['Poor\n0.4', 'Fair\n0.6', 'Good\n0.8', 'Excellent\n1.0'],
                       fontsize=8)

    for lbl, s, clr, ls in [('All',   sa, '#1f77b4', '-'),
                              ('Train', st, '#2ca02c', '--'),
                              ('Test',  se, '#d62728', ':')]:
        vals = [s['r2'], s['f5'], s['f10'], s['f15'], s['f20'], s['cov95']]
        vals += vals[:1]
        ax.plot(angles, vals, color=clr, lw=2.5, ls=ls, label=lbl)
        ax.fill(angles, vals, color=clr, alpha=0.08)
        # 数值标签
        for ang, val in zip(angles[:-1], vals[:-1]):
            ax.text(ang, val + 0.04, f'{val:.2f}',
                    ha='center', va='center', fontsize=8,
                    color=clr, fontweight='bold')

    ax.set_thetagrids(np.degrees(angles[:-1]), metrics, fontsize=12)
    ax.set_ylim(0, 1.1)
    ax.set_title(f'Performance Radar\n(All / Train / Test)',
                 fontsize=12, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.10), fontsize=10)

    # ── 右图：Train vs Test Gap 差距雷达 ─────────────────────
    ax = axes[1]
    vals_tr = [st['r2'], st['f5'], st['f10'], st['f15'], st['f20'], st['cov95']]
    vals_te = [se['r2'], se['f5'], se['f10'], se['f15'], se['f20'], se['cov95']]
    vals_tr_c = vals_tr + vals_tr[:1]
    vals_te_c = vals_te + vals_te[:1]

    ax.plot(angles, vals_tr_c, color='#2ca02c', lw=3, label='Train', zorder=4)
    ax.fill(angles, vals_tr_c, color='#2ca02c', alpha=0.12)
    ax.plot(angles, vals_te_c, color='#d62728', lw=3, ls='--', label='Test', zorder=4)
    ax.fill(angles, vals_te_c, color='#d62728', alpha=0.12)

    # 差距着色（Train>Test区域）
    for i in range(N):
        ang_start = angles[i]
        ang_end   = angles[i+1] if i+1 < N else angles[0] + 2*np.pi
        if vals_tr[i] > vals_te[i]:
            gap_vals = [0, vals_tr[i], vals_te[i], 0]
            gap_ang  = [ang_start, ang_start, ang_start, ang_start]
            ax.fill(gap_ang, gap_vals, color='orange', alpha=0.15)

    # 数值差标签
    for ang, vt, ve, met in zip(angles[:-1], vals_tr, vals_te, metrics):
        gap_v = vt - ve
        clr_g = '#d62728' if gap_v > 0.05 else ('#f58231' if gap_v > 0.02 else 'gray')
        ax.text(ang, max(vt, ve) + 0.08,
                f'Δ={gap_v:+.2f}', ha='center', va='center',
                fontsize=8, color=clr_g, fontweight='bold')

    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=9)
    ax.set_thetagrids(np.degrees(angles[:-1]), metrics, fontsize=12)
    ax.set_ylim(0, 1.2)
    ax.set_title('Train vs Test Gap Radar\n(Δ = Train − Test)',
                 fontsize=12, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.10), fontsize=10)

    fig.suptitle(f'Model Performance Radar  |  {mn}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(ind_dir, '08_Radar_Chart.png'),
                dpi=CONFIG['figure_dpi'], bbox_inches='tight', facecolor='white')
    plt.close()
    print("  ✓ 08_Radar_Chart.png")

    # ================================================================
    # 图9：分阶段误差柱状图  ★ 增强：颜色编码、目标线、差距标注、达标标记
    # ================================================================
    train_vals = [st[f'f{t}']*100 for t in thr_list]
    test_vals  = [se[f'f{t}']*100 for t in thr_list]
    all_vals   = [sa[f'f{t}']*100 for t in thr_list]
    x      = np.arange(len(thr_list))
    width  = 0.26
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    ax = axes[0]
    # 背景目标达成区着色
    target_line = 80  # 80%为达标线
    excellent_line = 90
    ax.axhspan(excellent_line, 110, alpha=0.06, color='green',  label='Excellent zone (≥90%)')
    ax.axhspan(target_line, excellent_line, alpha=0.06, color='limegreen', label='Good zone (80–90%)')
    ax.axhspan(60, target_line, alpha=0.06, color='orange',  label='Marginal zone (60–80%)')
    ax.axhspan(0, 60, alpha=0.04, color='red')

    b1 = ax.bar(x-width, train_vals, width, color='#2ca02c', alpha=0.82,
                edgecolor='black', lw=0.8, label='Train')
    b2 = ax.bar(x,       test_vals,  width, color='#d62728', alpha=0.82,
                edgecolor='black', lw=0.8, label='Test')
    b3 = ax.bar(x+width, all_vals,   width, color='#1f77b4', alpha=0.82,
                edgecolor='black', lw=0.8, label='All')

    for bars, vals in [(b1, train_vals), (b2, test_vals), (b3, all_vals)]:
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                    f'{v:.1f}%', ha='center', va='bottom',
                    fontsize=9, fontweight='bold')
            # 达标标记
            if v >= excellent_line:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2.5,
                        '★', ha='center', fontsize=10, color='darkgreen')

    for ref, ls_r, alpha_v, clr_r, lbl_r in [
        (90, '--', 0.8, 'darkgreen', '90% (Excellent)'),
        (80, '--', 0.7, 'green',     '80% (Target)'),
        (60, ':',  0.5, 'orange',    '60% (Minimum)'),
    ]:
        ax.axhline(ref, color=clr_r, ls=ls_r, lw=1.8, alpha=alpha_v,
                   label=lbl_r)

    ax.set_xticks(x)
    ax.set_xticklabels([f'F{t}%' for t in thr_list],
                       fontsize=12, fontweight='bold')
    ax.set_xlabel('Error Threshold',                   fontsize=13, fontweight='bold')
    ax.set_ylabel('Samples within Threshold (%)',      fontsize=13, fontweight='bold')
    ax.set_title('Prediction Accuracy by Error Threshold\n(★ = Excellent ≥90% | bg zones = performance levels)',
                 fontsize=12, fontweight='bold')
    ax.set_ylim(0, 115)
    ax.legend(fontsize=8.5, loc='lower right')
    ax.grid(True, alpha=0.3, axis='y')

    ax = axes[1]
    gaps      = [tr-te for tr, te in zip(train_vals, test_vals)]
    bar_cols  = ['#d62728' if g>10 else ('#f58231' if g>5 else '#2ca02c')
                 for g in gaps]
    bars_gap  = ax.bar(x, gaps, width=0.5, color=bar_cols, alpha=0.80,
                       edgecolor='black', lw=0.8)
    for bar, v, clr_b in zip(bars_gap, gaps, bar_cols):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.25,
                f'{v:+.1f}%', ha='center', va='bottom',
                fontsize=10, fontweight='bold', color=clr_b)
        # 过拟合程度标记
        if v > 10:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2.0,
                    'Overfit!', ha='center', fontsize=8.5,
                    color='darkred', style='italic')
        elif v > 5:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2.0,
                    'Mild gap', ha='center', fontsize=8.5,
                    color='darkorange', style='italic')
        else:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2.0,
                    'Good gen.', ha='center', fontsize=8.5,
                    color='darkgreen', style='italic')

    ax.axhline(0,  color='black', lw=1.5)
    ax.axhline(5,  color='#f58231', ls='--', lw=1.8, alpha=0.8,
               label='5% gap threshold')
    ax.axhline(10, color='#d62728', ls='--', lw=1.8, alpha=0.8,
               label='10% gap threshold (overfit risk)')
    ax.axhspan(0,   5,  alpha=0.07, color='green',  label='Good generalization')
    ax.axhspan(5,  10,  alpha=0.07, color='orange', label='Mild overfit')
    ax.axhspan(10, max(max(gaps)*1.4, 20), alpha=0.07, color='red',
               label='Overfit risk')
    ax.set_xticks(x)
    ax.set_xticklabels([f'F{t}%' for t in thr_list],
                       fontsize=12, fontweight='bold')
    ax.set_xlabel('Error Threshold',            fontsize=13, fontweight='bold')
    ax.set_ylabel('Train − Test Gap (%)',        fontsize=13, fontweight='bold')
    ax.set_title('Generalization Gap by Error Threshold\n(Red=overfit risk, Orange=mild gap, Green=good)',
                 fontsize=12, fontweight='bold')
    ax.set_ylim(0, max(max(gaps)*1.45, 20))
    ax.legend(fontsize=8.5, loc='upper right'); ax.grid(True, alpha=0.3, axis='y')

    fig.suptitle(f'Error Threshold Analysis  |  {mn}\n'
                 f'Test F5={se["f5"]*100:.1f}%  F10={se["f10"]*100:.1f}%  '
                 f'F15={se["f15"]*100:.1f}%  F20={se["f20"]*100:.1f}%',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(ind_dir, '09_Threshold_Bar_Chart.png'),
                dpi=CONFIG['figure_dpi'], bbox_inches='tight', facecolor='white')
    plt.close()
    print("  ✓ 09_Threshold_Bar_Chart.png")

    # ================================================================
    # 图10：误差箱线图  ★ 增强：均值±std标注、样本量、分位数数值、小提琴
    # ================================================================
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    data_groups   = [st['relative_errors'], se['relative_errors'],
                     sa['relative_errors']]
    group_labels  = [f'Train\n(n={len(st["relative_errors"])})',
                     f'Test\n(n={len(se["relative_errors"])})',
                     f'All\n(n={len(sa["relative_errors"])})']
    group_colors  = ['#2ca02c', '#d62728', '#1f77b4']

    ax = axes[0]
    # 小提琴底图（显示分布形状）
    parts = ax.violinplot(data_groups, positions=[1, 2, 3],
                          showmedians=False, showextrema=False)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(group_colors[i])
        pc.set_alpha(0.20)
        pc.set_edgecolor(group_colors[i])

    bp = ax.boxplot(data_groups, positions=[1, 2, 3],
                    labels=group_labels, patch_artist=True,
                    showmeans=True, showfliers=True,
                    medianprops=dict(color='black', lw=2.5),
                    meanprops=dict(marker='D', markerfacecolor='white',
                                   markeredgecolor='black', markersize=8, lw=0),
                    flierprops=dict(marker='o', markersize=5, alpha=0.6, lw=0.5),
                    whiskerprops=dict(lw=1.5), capprops=dict(lw=2),
                    boxprops=dict(lw=1.5))
    for patch, color in zip(bp['boxes'], group_colors):
        patch.set_facecolor(color); patch.set_alpha(0.50)
    for flier, color in zip(bp['fliers'], group_colors):
        flier.set_markerfacecolor(color); flier.set_markeredgecolor(color)

    # 均值±std标注
    for pos, data, clr in zip([1, 2, 3], data_groups, group_colors):
        mn_d  = data.mean()
        std_d = data.std()
        q1_d  = np.percentile(data, 25)
        q3_d  = np.percentile(data, 75)
        med_d = np.median(data)
        ax.text(pos, mn_d + std_d * 1.05,
                f'μ={mn_d:.1f}%\nσ={std_d:.1f}%',
                ha='center', va='bottom', fontsize=8, color=clr,
                fontweight='bold')
        # Q1/Q3/Median 数值
        ax.text(pos + 0.28, q1_d,  f'Q1={q1_d:.1f}', ha='left',
                fontsize=7.5, color=clr, alpha=0.85)
        ax.text(pos + 0.28, med_d, f'Med={med_d:.1f}', ha='left',
                fontsize=7.5, color='black', fontweight='bold')
        ax.text(pos + 0.28, q3_d,  f'Q3={q3_d:.1f}', ha='left',
                fontsize=7.5, color=clr, alpha=0.85)

    for ref, ls_r, clr_r in [(10,'--','gray'), (20,':','gray')]:
        ax.axhline(ref, color=clr_r, ls=ls_r, lw=1.5, alpha=0.7,
                   label=f'{ref}% reference')
    ax.set_ylabel('Relative Error (%)', fontsize=13, fontweight='bold')
    ax.set_title('Error Distribution  (Violin + Box)\n'
                 '(◆=Mean, ─=Median, text=μ±σ and Q1/Q3)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3, axis='y')

    ax = axes[1]
    data_clipped = [d[d <= 30] for d in data_groups]
    labels_clip  = [f'Train\n(≤30%, n={len(data_clipped[0])})',
                    f'Test\n(≤30%, n={len(data_clipped[1])})',
                    f'All\n(≤30%, n={len(data_clipped[2])})']
    # 小提琴
    parts2 = ax.violinplot(data_clipped, positions=[1, 2, 3],
                           showmedians=False, showextrema=False)
    for i, pc in enumerate(parts2['bodies']):
        pc.set_facecolor(group_colors[i])
        pc.set_alpha(0.18)
        pc.set_edgecolor(group_colors[i])

    bp2 = ax.boxplot(data_clipped, positions=[1, 2, 3],
                     labels=labels_clip, patch_artist=True,
                     showmeans=True, showfliers=True,
                     medianprops=dict(color='black', lw=2.5),
                     meanprops=dict(marker='D', markerfacecolor='white',
                                    markeredgecolor='black', markersize=8, lw=0),
                     flierprops=dict(marker='o', markersize=5, alpha=0.6, lw=0.5),
                     whiskerprops=dict(lw=1.5), capprops=dict(lw=2),
                     boxprops=dict(lw=1.5))
    for patch, color in zip(bp2['boxes'], group_colors):
        patch.set_facecolor(color); patch.set_alpha(0.50)
    for flier, color in zip(bp2['fliers'], group_colors):
        flier.set_markerfacecolor(color); flier.set_markeredgecolor(color)

    # 均值±std标注
    for pos, data, clr in zip([1, 2, 3], data_clipped, group_colors):
        if len(data) == 0:
            continue
        mn_d  = data.mean()
        std_d = data.std()
        ax.text(pos, mn_d + std_d + 0.5,
                f'μ={mn_d:.1f}\nσ={std_d:.1f}',
                ha='center', va='bottom', fontsize=8, color=clr,
                fontweight='bold')

    ax.axhspan(0,  5,  alpha=0.08, color='green',     label='Excellent (0–5%)')
    ax.axhspan(5,  10, alpha=0.08, color='limegreen',  label='Good (5–10%)')
    ax.axhspan(10, 20, alpha=0.08, color='orange',     label='Acceptable (10–20%)')
    ax.axhspan(20, 30, alpha=0.08, color='red',        label='Poor (20–30%)')
    for ref in [5, 10, 20]:
        ax.axhline(ref, color='gray', ls='--', lw=1.2, alpha=0.6)
        ax.text(3.55, ref, f'{ref}%', fontsize=8.5, va='center', color='gray')
    ax.set_ylabel('Relative Error (%)', fontsize=13, fontweight='bold')
    ax.set_title('Error Box Plot (Clipped ≤30%)  +  Violin\nwith Accuracy Zone Shading  (μ=Mean, σ=Std)',
                 fontsize=12, fontweight='bold')
    ax.set_ylim(0, 33)
    ax.legend(fontsize=8.5, loc='upper right'); ax.grid(True, alpha=0.3, axis='y')

    fig.suptitle(f'Relative Error Box Plot  |  {mn}\n'
                 f'Train: median={np.median(st["relative_errors"]):.2f}%  '
                 f'mean={st["mean_rel_err"]:.2f}%  |  '
                 f'Test: median={np.median(se["relative_errors"]):.2f}%  '
                 f'mean={se["mean_rel_err"]:.2f}%',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(ind_dir, '10_Error_Boxplot.png'),
                dpi=CONFIG['figure_dpi'], bbox_inches='tight', facecolor='white')
    plt.close()
    print("  ✓ 10_Error_Boxplot.png")

    print(f"\n✓ 全部10张单独分析图已保存至 individual_models/")


# ====================== 统计报告 ======================
def create_summary_statistics(bundle, sa, st, se, out_dir):
    print("\n=== 创建汇总统计报告 ===")
    stats_dir  = os.path.join(out_dir, 'statistics')
    model_name = bundle['model_name']

    rows = []
    for lbl, s in [('All', sa), ('Train', st), ('Test', se)]:
        rel = s['relative_errors']
        rows.append({
            'Dataset': lbl, 'Model': model_name,
            'N_Samples': len(s['y_true']),
            'R2': s['r2'], 'RMSE': s['rmse'], 'MAE': s['mae'],
            'Mean_Rel_Error': s['mean_rel_err'],
            'Median_Rel_Error': s['median_rel_err'],
            'F5': s['f5'], 'F10': s['f10'], 'F15': s['f15'],
            'F20': s['f20'], 'F25': s['f25'], 'F30': s['f30'],
            'Exceed_F10_%': (1-s['f10'])*100,
            'Exceed_F20_%': (1-s['f20'])*100,
            'GP_Cov95': s['cov95'], 'GP_Cov68': s['cov68'],
            'Error_Skewness': float(pd.Series(rel).skew()),
            'Error_Kurtosis': float(pd.Series(rel).kurt()),
            'Gini_Coefficient': compute_gini(s['absolute_errors']),
        })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(stats_dir, 'detailed_model_statistics.csv'),
              index=False, encoding='utf-8')

    txt = os.path.join(stats_dir, 'ranking_statistics.txt')
    with open(txt, 'w', encoding='utf-8') as f:
        f.write("=" * 70 + "\n")
        f.write("TOP1 高斯过程模型 CDF 分析报告 v5（增强注释版）\n")
        f.write("=" * 70 + "\n\n")
        f.write(f"模型: {model_name}\n")
        f.write(f"时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        for lbl, s in [('全部数据', sa), ('训练集', st), ('测试集', se)]:
            f.write(f"--- {lbl} ---\n")
            f.write(f"  N={len(s['y_true'])}  R²={s['r2']:.4f}  "
                    f"RMSE={s['rmse']:.4f}  MAE={s['mae']:.4f}\n")
            f.write(f"  MRE={s['mean_rel_err']:.2f}%  "
                    f"MedRE={s['median_rel_err']:.2f}%\n")
            f.write(f"  F5={s['f5']:.3f}  F10={s['f10']:.3f}  "
                    f"F15={s['f15']:.3f}  F20={s['f20']:.3f}\n")
            f.write(f"  Exceedance@F10={(1-s['f10'])*100:.1f}%  "
                    f"@F20={(1-s['f20'])*100:.1f}%\n")
            f.write(f"  Gini={compute_gini(s['absolute_errors']):.3f}\n\n")
        f.write("\n完整图片列表（10张，v5增强版）:\n")
        for line in [
            "  01_CDF_All_Train_Test.png         ← 分区着色+关键百分位+统计框",
            "  02_Prediction_Scatter.png         ← ±5/10/20%误差带+误差着色+离群标注",
            "  03_Exceedance_Probability.png     ← 达标/警戒/危险分区+关键点标注",
            "  04_Error_PDF.png                  ← 峰值★+IQR阴影+KS检验+交叉点标注",
            "  05_Cumulative_Error_Contribution.png ← 基尼系数+帕累托线+关键点标注",
            "  06_Error_Quantile_Plot.png        ← ±30%带+Shapiro检验+尾部着色",
            "  07_Error_Histograms.png           ← 分区着色+KDE+指数拟合+统计框",
            "  08_Radar_Chart.png                ← 数值标签+等级背景+差距雷达",
            "  09_Threshold_Bar_Chart.png        ← 达标★+过拟合标记+背景色",
            "  10_Error_Boxplot.png              ← 小提琴+均值σ标注+Q1/Q3数值",
        ]:
            f.write(line + "\n")
    print(f"✓ 统计报告已保存")
    return df


# ====================== 主函数 ======================
def main():
    print("=" * 70)
    print("TOP1 高斯过程模型 CDF 完整分析 v5（增强可视化注释版）")
    print("所有10张图片均增加了更丰富的注释、分区着色和统计信息")
    print("=" * 70)
    print(f"输出路径: {CONFIG['output_path']}")
    print(f"时间戳  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)

    out_dir = setup_output_directory()
    bundle  = load_gp_model_and_data()
    sa, st, se = calculate_errors(bundle)

    save_data_files(bundle, sa, st, se, out_dir)
    create_comprehensive_cdf_plot(bundle, sa, st, se, out_dir)
    create_individual_model_plots(bundle, sa, st, se, out_dir)
    create_summary_statistics(bundle, sa, st, se, out_dir)

    with open(os.path.join(out_dir, 'analysis_config.txt'),
              'w', encoding='utf-8') as f:
        f.write("=== CDF分析配置 v5 ===\n")
        f.write(f"模型pkl  : {CONFIG['pkl_file']}\n")
        f.write(f"数据文件 : {CONFIG['data_file']}\n")
        f.write(f"最大误差 : {CONFIG['max_error']}%\n")
        f.write(f"Bootstrap: {CONFIG['bootstrap_n']}次 "
                f"{int(CONFIG['bootstrap_ci']*100)}%CI\n")
        f.write(f"分析时间 : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    print("\n" + "=" * 70)
    print("✅ CDF分析 v5 完成！")
    print("=" * 70)
    mn = bundle['model_name']
    print(f"  模型: {mn}")
    print(f"  Test R²={se['r2']:.4f}  F10={se['f10']*100:.1f}%  "
          f"F20={se['f20']*100:.1f}%")
    print(f"\n  📊 v5 增强内容总结:")
    for line in [
        "  01_CDF: 分区着色 + CDF值标注 + 统计文本框 + 水平百分位线",
        "  02_Scatter: ±5/10/20%误差带 + 误差颜色映射 + 离群点标注",
        "  03_Exceedance: 4区着色(Safe/Warning/Caution/Danger) + 关键点标注",
        "  04_PDF: 峰值★标注 + IQR阴影 + KS检验 + 曲线交叉点标注",
        "  05_CumError: 基尼系数 + 帕累托80/20参考线 + 关键比例点标注",
        "  06_Quantile: ±30%置信带 + Shapiro-Wilk检验 + 尾部着色",
        "  07_Histogram: 分区颜色 + KDE曲线 + 指数拟合 + 完整统计框",
        "  08_Radar: 数值标签 + 4级背景环 + 差距Δ雷达图",
        "  09_ThresholdBar: 背景色分区 + ★达标标记 + 过拟合文字标注",
        "  10_Boxplot: 小提琴图底层 + μ±σ标注 + Q1/Med/Q3数值显示",
    ]:
        print(line)


if __name__ == "__main__":
    main()

TOP1 高斯过程模型 CDF 完整分析 v5（增强可视化注释版）
所有10张图片均增加了更丰富的注释、分区着色和统计信息
输出路径: D:\博二上\模型分析可视化\CDF分析6
时间戳  : 2026-04-13 19:16:03
✓ 输出目录已创建: D:\博二上\模型分析可视化\CDF分析6
=== 加载TOP1高斯过程模型和数据 ===
✓ 模型类型  : GP_Matern_0.5
✓ 特征列表  : ['deltaP_热导率W/(mk)', 'x', 'Ec', 'Ω', 'deltaP_E13 Electron affinity', 'ΔSmix']
✓ Test R²   : 0.759772415859425
✓ 有效样本  : 202
✓ 训练集: 161  测试集: 41

=== 计算预测误差 ===
  全部数据  R²=0.8505  MAE=0.9140  MRE=9.50%  F10=0.649
  训练集  R²=0.8748  MAE=0.8392  MRE=8.97%  F10=0.683
  测试集  R²=0.7598  MAE=1.2080  MRE=11.60%  F10=0.512

=== 保存数据文件 ===
✓ CSV数据已保存
  生成Origin可用Excel文件...
    ✓ 01_CDF_Data.xlsx
    ✓ 02_Radar_Chart_Data.xlsx
    ✓ 03_Error_Histograms.xlsx
    ✓ 04_Prediction_Scatter.xlsx
    ✓ 05_GP_Uncertainty.xlsx
    ✓ 06_Threshold_Bar_Data.xlsx
    ✓ 07_Boxplot_Data.xlsx
    ✓ 08_Exceedance_Data.xlsx
    ✓ 09_Error_PDF_Data.xlsx
    ✓ 10_Cumulative_Error_Data.xlsx
    ✓ 11_Error_Quantile_Data.xlsx
    ✓ 00_Origin_Plotting_Guide.xlsx
    ✅ 所有Origin Excel文件已生成！

=== 创建综合CDF分析图 ===
✓ 综合CDF图